# 📘 Pipeline Orchestration & Workflows
## Databricks Notebook — Production Pipeline Management

**What you'll learn:**
- Databricks Workflows (Lakeflow Jobs)
- Manual orchestration: DAG executor, dependency resolution
- Retry & error handling patterns
- Idempotent pipeline design
- Apache Airflow concepts
- Scheduling, metadata tracking, multi-pipeline architecture

**Prerequisites:** Notebooks 01-24

---

---
# 📗 Section 1: Why Orchestration Matters

**A script is NOT a pipeline.** A production pipeline needs:

| Feature | Script | Production Pipeline |
|---|---|---|
| Dependencies | Manual ordering | Automatic DAG resolution |
| Failures | Crashes | Retry, alert, recover |
| Scheduling | Manual trigger | Cron, event-driven |
| Monitoring | Print statements | Metrics, dashboards, alerts |
| Idempotency | Breaks on re-run | Safe to re-run anytime |

## Pipeline Lifecycle

```
DEFINE → SCHEDULE → EXECUTE → MONITOR → ALERT → RECOVER
  │         │          │          │         │        │
  DAG     Cron/     Run tasks   Track    Notify   Retry/
  deps    Event     in order    metrics   team    Skip
```

## Orchestration Concepts -- Deep Dive

### What is Orchestration?

Orchestration coordinates WHEN and in WHAT ORDER pipelines run:

```
WITHOUT ORCHESTRATION:                  WITH ORCHESTRATION:
┌─────────────────────────┐            ┌─────────────────────────┐
│ Run manually            │            │ Scheduled automatically  │
│ Hope dependencies work  │            │ Dependencies enforced    │
│ No retry on failure     │            │ Auto-retry on failure    │
│ No visibility           │            │ Full monitoring/alerting │
│ "Did it run last night?"│            │ Dashboard shows status   │
└─────────────────────────┘            └─────────────────────────┘
```

### Key Orchestration Concepts

| Concept | Definition | Example |
|---------|-----------|---------|
| **DAG** | Directed Acyclic Graph -- tasks with dependencies | ingest → transform → load |
| **Task** | A single unit of work | Run a notebook, execute SQL |
| **Dependency** | Task B runs only after Task A succeeds | silver depends on bronze |
| **Schedule** | When the DAG runs | Every day at 6 AM |
| **Trigger** | What starts a run | Schedule, file arrival, API call |
| **Retry** | Re-run failed tasks automatically | 3 retries with 5-min delay |
| **SLA** | Maximum acceptable runtime | Must complete by 8 AM |
| **Backfill** | Re-run for historical dates | Reprocess last 30 days |

### DAG Design Principles

**1. Idempotency** -- Safe to rerun
```python
# Bad: Appends every time (duplicates on retry!)
df.write.mode("append").saveAsTable("silver.orders")

# Good: Overwrites partition (safe to retry)
df.write.mode("overwrite").option("replaceWhere", f"date='{date}'").saveAsTable("silver.orders")
```

**2. Atomicity** -- All or nothing
```python
# Bad: Partial writes leave table in inconsistent state
write_part1()
write_part2()  # If this fails, part1 is already written!

# Good: Use transactions
with spark.sql("BEGIN TRANSACTION"):
    write_part1()
    write_part2()
```

**3. Observability** -- Know what's happening
```python
# Log progress at each step
logger.info(f"Processing {date}: {record_count} records")
logger.info(f"Quality checks: {passed}/{total} passed")
logger.info(f"Written to {target_table}: {written_count} rows")
```

---
# 📗 Section 2: Databricks Workflows (Lakeflow Jobs)

**Databricks Workflows** (now called Lakeflow Jobs) provide UI-based orchestration.

## Task Types
- **Notebook task:** Run a Databricks notebook
- **SQL task:** Execute a SQL query
- **Python script:** Run a .py file
- **DLT pipeline:** Trigger a Declarative Pipeline
- **JAR task:** Run a Java/Scala JAR

## DAG Example

```
┌─────────────┐     ┌──────────────┐     ┌─────────────┐
│ ingest_raw  │────▶│ clean_silver │────▶│ build_gold  │
└─────────────┘     └──────────────┘     └─────────────┘
       │                                        │
       │            ┌──────────────┐            │
       └───────────▶│ quality_check│◀───────────┘
                    └──────────────┘
                           │
                    ┌──────────────┐
                    │ send_report  │
                    └──────────────┘
```

## Workflow JSON Config (what the UI generates)
```json
{
  "name": "daily_etl_pipeline",
  "tasks": [
    {"task_key": "ingest", "notebook_task": {"notebook_path": "/pipelines/01_ingest"}},
    {"task_key": "clean", "depends_on": [{"task_key": "ingest"}], "notebook_task": {"notebook_path": "/pipelines/02_clean"}},
    {"task_key": "gold", "depends_on": [{"task_key": "clean"}], "notebook_task": {"notebook_path": "/pipelines/03_gold"}}
  ],
  "schedule": {"quartz_cron_expression": "0 0 6 * * ?", "timezone_id": "UTC"}
}
```

## Lakeflow Jobs -- Complete Reference

### Job Configuration

```yaml
# In databricks.yml (DABs bundle):
resources:
  jobs:
    daily_pipeline:
      name: "Daily Revenue Pipeline"
      
      # Schedule
      schedule:
        quartz_cron_expression: "0 0 6 * * ?"  # 6 AM daily
        timezone_id: "America/New_York"
        pause_status: UNPAUSED
      
      # Email notifications
      email_notifications:
        on_failure: ["data-team@company.com"]
        on_success: []
        on_start: []
      
      # Timeout
      timeout_seconds: 7200  # 2 hours max
      
      # Tasks
      tasks:
        - task_key: ingest_orders
          notebook_task:
            notebook_path: ./src/ingest_orders
          new_cluster:
            spark_version: "14.3.x-scala2.12"
            num_workers: 4
          retry_on_timeout: true
          max_retries: 3
          min_retry_interval_millis: 300000  # 5 minutes
        
        - task_key: ingest_customers
          notebook_task:
            notebook_path: ./src/ingest_customers
          new_cluster:
            spark_version: "14.3.x-scala2.12"
            num_workers: 2
          # Runs in PARALLEL with ingest_orders (no depends_on)
        
        - task_key: transform_silver
          depends_on:
            - task_key: ingest_orders
            - task_key: ingest_customers
          notebook_task:
            notebook_path: ./src/transform_silver
          new_cluster:
            spark_version: "14.3.x-scala2.12"
            num_workers: 8
        
        - task_key: build_gold
          depends_on:
            - task_key: transform_silver
          sql_task:
            query: "CALL gold.build_daily_revenue()"
            warehouse_id: "abc123"
        
        - task_key: quality_checks
          depends_on:
            - task_key: build_gold
          notebook_task:
            notebook_path: ./src/quality_checks
          new_cluster:
            spark_version: "14.3.x-scala2.12"
            num_workers: 2
        
        - task_key: notify_success
          depends_on:
            - task_key: quality_checks
          notebook_task:
            notebook_path: ./src/send_notification
          new_cluster:
            spark_version: "14.3.x-scala2.12"
            num_workers: 1
```

### Task Types in Lakeflow Jobs

| Task Type | Use For | Example |
|-----------|---------|---------|
| **Notebook** | Python/SQL notebooks | Data transformation |
| **Python script** | .py files in Repos | Utility scripts |
| **SQL** | SQL queries on warehouse | Aggregations, DDL |
| **DLT pipeline** | Declarative pipelines | Medallion pipeline |
| **dbt** | dbt models | SQL transformations |
| **JAR** | Java/Scala applications | Legacy code |
| **Spark Python** | PySpark scripts | Large-scale processing |

In [ ]:
# Lakeflow Jobs simulation with dependency resolution
from datetime import datetime
import time

class LakeflowJobRunner:
    """Simulates Lakeflow Jobs execution with dependencies."""
    
    def __init__(self, job_name):
        self.job_name = job_name
        self.tasks = {}
        self.run_history = []
    
    def add_task(self, key, task_type, depends_on=None, max_retries=3, timeout_min=60):
        self.tasks[key] = {
            "type": task_type,
            "depends_on": depends_on or [],
            "max_retries": max_retries,
            "timeout_min": timeout_min,
            "status": "pending",
            "attempts": 0,
            "duration_sec": 0,
        }
    
    def _get_execution_order(self):
        """Topological sort -- determines parallel execution groups."""
        executed = set()
        groups = []
        
        while len(executed) < len(self.tasks):
            ready = []
            for key, task in self.tasks.items():
                if key not in executed:
                    if all(dep in executed for dep in task["depends_on"]):
                        ready.append(key)
            if not ready:
                break  # Circular dependency or all done
            groups.append(ready)
            executed.update(ready)
        
        return groups
    
    def run(self, simulate_failure=None):
        """Execute the job."""
        start_time = datetime.now()
        print(f"\n🚀 JOB: {self.job_name}")
        print(f"   Started: {start_time.strftime('%H:%M:%S')}")
        
        groups = self._get_execution_order()
        print(f"   Execution plan ({len(groups)} stages):")
        for i, group in enumerate(groups):
            print(f"     Stage {i+1}: {' || '.join(group)} (parallel)")
        
        print()
        job_success = True
        
        for stage_num, group in enumerate(groups):
            print(f"   --- Stage {stage_num + 1} ---")
            
            for task_key in group:
                task = self.tasks[task_key]
                
                # Simulate task execution with retry
                success = False
                for attempt in range(task["max_retries"] + 1):
                    task["attempts"] = attempt + 1
                    
                    # Simulate failure for specific task
                    if simulate_failure and task_key == simulate_failure and attempt < 2:
                        print(f"   ❌ {task_key} (attempt {attempt+1}): FAILED -- retrying in 5s...")
                        task["status"] = "retrying"
                        continue
                    
                    # Success
                    task["status"] = "succeeded"
                    task["duration_sec"] = 10 + len(task_key) * 2  # Simulate duration
                    success = True
                    
                    attempts_str = f" (after {attempt+1} attempts)" if attempt > 0 else ""
                    print(f"   ✅ {task_key} ({task['type']}): {task['duration_sec']}s{attempts_str}")
                    break
                
                if not success:
                    task["status"] = "failed"
                    job_success = False
                    print(f"   ❌ {task_key}: FAILED after {task['max_retries']+1} attempts")
                    print(f"   ⛔ Downstream tasks skipped")
                    break
            
            if not job_success:
                break
        
        end_time = datetime.now()
        duration = (end_time - start_time).total_seconds()
        
        print(f"\n   {'✅ JOB SUCCEEDED' if job_success else '❌ JOB FAILED'}")
        print(f"   Duration: {duration:.1f}s")
        return job_success


# Build and run a realistic pipeline
job = LakeflowJobRunner("daily_revenue_pipeline")
job.add_task("ingest_orders", "notebook", max_retries=3)
job.add_task("ingest_customers", "notebook", max_retries=3)
job.add_task("ingest_products", "notebook", max_retries=3)
job.add_task("transform_silver", "notebook", depends_on=["ingest_orders", "ingest_customers", "ingest_products"])
job.add_task("build_gold", "sql", depends_on=["transform_silver"])
job.add_task("quality_checks", "notebook", depends_on=["build_gold"])
job.add_task("send_notification", "notebook", depends_on=["quality_checks"])

print("=" * 60)
print("SCENARIO 1: Normal run (all succeed)")
job.run()

print("\n" + "=" * 60)
print("SCENARIO 2: ingest_orders fails twice, then succeeds")
# Reset statuses
for task in job.tasks.values():
    task["status"] = "pending"
    task["attempts"] = 0
job.run(simulate_failure="ingest_orders")


---
# 📗 Section 3: Manual Orchestration in Python

In [ ]:
import time
from datetime import datetime
from collections import defaultdict, deque

class DAGExecutor:
    """Execute tasks in dependency order with status tracking."""
    
    def __init__(self, name):
        self.name = name
        self.tasks = {}
        self.dependencies = defaultdict(list)
        self.results = {}
    
    def add_task(self, task_id, func, depends_on=None):
        """Register a task with its dependencies."""
        self.tasks[task_id] = func
        if depends_on:
            self.dependencies[task_id] = depends_on
    
    def _topological_sort(self):
        """Resolve execution order using topological sort."""
        in_degree = {t: 0 for t in self.tasks}
        for task, deps in self.dependencies.items():
            in_degree[task] = len(deps)
        
        queue = deque([t for t, d in in_degree.items() if d == 0])
        order = []
        
        while queue:
            task = queue.popleft()
            order.append(task)
            for t, deps in self.dependencies.items():
                if task in deps:
                    in_degree[t] -= 1
                    if in_degree[t] == 0:
                        queue.append(t)
        
        if len(order) != len(self.tasks):
            raise ValueError("Circular dependency detected!")
        return order
    
    def run(self):
        """Execute all tasks in dependency order."""
        order = self._topological_sort()
        print(f"{'='*50}")
        print(f"DAG: {self.name}")
        print(f"Execution order: {' → '.join(order)}")
        print(f"{'='*50}")
        
        for task_id in order:
            # Check dependencies passed
            deps = self.dependencies.get(task_id, [])
            deps_ok = all(self.results.get(d, {}).get("status") == "success" for d in deps)
            
            if not deps_ok:
                self.results[task_id] = {"status": "skipped", "reason": "dependency failed"}
                print(f"  ⏭️ {task_id}: SKIPPED (dependency failed)")
                continue
            
            start = time.time()
            try:
                result = self.tasks[task_id]()
                duration = time.time() - start
                self.results[task_id] = {"status": "success", "duration": round(duration, 2), "result": result}
                print(f"  ✅ {task_id}: SUCCESS ({duration:.2f}s)")
            except Exception as e:
                duration = time.time() - start
                self.results[task_id] = {"status": "failed", "duration": round(duration, 2), "error": str(e)}
                print(f"  ❌ {task_id}: FAILED ({e})")
        
        # Summary
        success = sum(1 for r in self.results.values() if r["status"] == "success")
        failed = sum(1 for r in self.results.values() if r["status"] == "failed")
        print(f"{'='*50}")
        print(f"Results: {success} success, {failed} failed, {len(self.results)-success-failed} skipped")
        return self.results

# Define and run a pipeline DAG
dag = DAGExecutor("daily_etl")
dag.add_task("ingest_orders", lambda: spark.table("techmart_dw.orders").count())
dag.add_task("ingest_customers", lambda: spark.table("techmart_dw.customers").count())
dag.add_task("silver_orders", lambda: "cleaned", depends_on=["ingest_orders"])
dag.add_task("silver_customers", lambda: "cleaned", depends_on=["ingest_customers"])
dag.add_task("gold_dashboard", lambda: "built", depends_on=["silver_orders", "silver_customers"])
dag.add_task("send_report", lambda: "sent", depends_on=["gold_dashboard"])

dag.run()

---
# 📗 Section 4: Retry & Error Handling

In [ ]:
import time
import random

def retry_task(func, task_name, max_retries=3, base_delay=1.0):
    """Execute a task with exponential backoff retry."""
    for attempt in range(1, max_retries + 1):
        try:
            result = func()
            print(f"  ✅ {task_name}: success (attempt {attempt})")
            return result
        except Exception as e:
            if attempt == max_retries:
                print(f"  ❌ {task_name}: FAILED after {max_retries} attempts — {e}")
                raise
            delay = base_delay * (2 ** (attempt - 1)) + random.uniform(0, 0.5)
            print(f"  ⚠️ {task_name}: attempt {attempt} failed ({e}), retrying in {delay:.1f}s...")
            time.sleep(delay)

# Demo: flaky task that fails sometimes
call_count = [0]
def flaky_task():
    call_count[0] += 1
    if call_count[0] < 3:
        raise ConnectionError("Database timeout")
    return "data loaded"

print("Retry demo:")
result = retry_task(flaky_task, "load_data", max_retries=5, base_delay=0.1)
print(f"Result: {result}")

## Retry and Error Handling -- Production Patterns

### Retry Strategy

Not all errors should be retried. Categorize errors first:

| Error Type | Retry? | Example |
|-----------|--------|---------|
| **Transient** | Yes | Network timeout, rate limit, temporary DB unavailability |
| **Permanent** | No | Bad data, schema mismatch, permission denied |
| **Resource** | Yes (with more resources) | OOM, disk full |
| **Logic** | No | Division by zero, null pointer |

### Exponential Backoff Pattern

```
Attempt 1: immediate
Attempt 2: wait 1 second
Attempt 3: wait 2 seconds
Attempt 4: wait 4 seconds (give up)
```

### Dead Letter Queue (DLQ)

For records that fail after all retries, send to a DLQ for manual review:
- Never silently drop failed records
- DLQ enables investigation and reprocessing
- Alert data team when DLQ grows beyond threshold

In [ ]:
# Retry and DLQ patterns
import time
from datetime import datetime

class TransientError(Exception):
    pass

class PermanentError(Exception):
    pass

def retry_with_backoff(func, max_retries=3, base_delay=1, backoff_factor=2):
    """Retry with exponential backoff. Only retries TransientError."""
    for attempt in range(max_retries + 1):
        try:
            return func()
        except TransientError as e:
            if attempt == max_retries:
                raise
            delay = base_delay * (backoff_factor ** attempt)
            print(f"   Attempt {attempt+1} failed: {e}. Retrying in {delay}s...")
        except PermanentError:
            raise  # Never retry permanent errors

def process_batch_with_dlq(records, process_fn, dlq_store):
    """Process records, send failures to DLQ."""
    successes = []
    dlq_store_local = dlq_store

    for record in records:
        try:
            result = process_fn(record)
            successes.append(result)
        except PermanentError as e:
            dlq_store_local.append({
                "record": record,
                "error": str(e),
                "failed_at": datetime.now().isoformat(),
                "type": "permanent"
            })
        except TransientError as e:
            dlq_store_local.append({
                "record": record,
                "error": str(e),
                "failed_at": datetime.now().isoformat(),
                "type": "transient_exhausted"
            })

    return successes, dlq_store_local


# Demo
print("🔄 RETRY AND DLQ DEMO")
print("=" * 60)

attempt_counter = {"n": 0}

def flaky_api_call():
    attempt_counter["n"] += 1
    if attempt_counter["n"] <= 2:
        raise TransientError("Connection timeout")
    return {"status": "success", "data": [1, 2, 3]}

print("\n1. Retry with backoff:")
result = retry_with_backoff(flaky_api_call, max_retries=3, base_delay=0)
print(f"   Result: {result} (after {attempt_counter['n']} attempts)")

print("\n2. Batch processing with DLQ:")
records = [
    {"id": 1, "amount": 100},
    {"id": 2, "amount": None},   # Will fail permanently
    {"id": 3, "amount": 200},
    {"id": 4, "amount": -50},    # Will fail permanently
    {"id": 5, "amount": 300},
]

dlq = []

def process_record(r):
    if r["amount"] is None:
        raise PermanentError("NULL amount")
    if r["amount"] < 0:
        raise PermanentError("Negative amount")
    return {"id": r["id"], "processed_amount": r["amount"] * 1.08}

successes, dlq = process_batch_with_dlq(records, process_record, dlq)
print(f"   Successes: {len(successes)}")
print(f"   DLQ: {len(dlq)} records")
for item in dlq:
    print(f"      Record {item['record']['id']}: {item['error']}")


---
# 📗 Section 5: Idempotent Pipeline Design

**Idempotent** = running N times produces the same result as running once.

| Pattern | How | When |
|---|---|---|
| MERGE (upsert) | INSERT new, UPDATE existing | Incremental loads |
| Partition overwrite | Overwrite specific partition | Date-partitioned tables |
| Full overwrite | DROP + recreate | Small reference tables |
| Checkpoint | Skip already-processed data | Streaming/file ingestion |

In [ ]:
%sql
-- MERGE-based idempotency: safe to run multiple times
USE techmart_dw;

-- This MERGE is idempotent: running it twice produces the same result
MERGE INTO orders AS target
USING (SELECT * FROM orders WHERE order_id <= 10) AS source
ON target.order_id = source.order_id
WHEN MATCHED THEN UPDATE SET target.updated_at = current_timestamp()
WHEN NOT MATCHED THEN INSERT *;

-- Verify: run again — no change
SELECT COUNT(*) AS order_count FROM orders WHERE order_id <= 10;

---
# 📗 Section 6: Apache Airflow Concepts (Conceptual)

⚠️ Airflow requires separate infrastructure — patterns shown conceptually.

```python
# Airflow DAG definition (CONCEPTUAL)
from airflow import DAG
from airflow.operators.python import PythonOperator
from airflow.providers.databricks.operators.databricks import DatabricksRunNowOperator
from datetime import datetime, timedelta

default_args = {
    'owner': 'data_engineering',
    'retries': 3,
    'retry_delay': timedelta(minutes=5),
    'email_on_failure': True,
}

with DAG(
    'daily_etl_pipeline',
    default_args=default_args,
    schedule_interval='0 6 * * *',  # Daily at 6 AM UTC
    start_date=datetime(2024, 1, 1),
    catchup=False,
) as dag:
    
    ingest = DatabricksRunNowOperator(
        task_id='ingest_raw',
        databricks_conn_id='databricks_default',
        job_id=12345,
    )
    
    transform = DatabricksRunNowOperator(
        task_id='transform_silver',
        databricks_conn_id='databricks_default',
        job_id=12346,
    )
    
    ingest >> transform  # dependency
```

**Key Airflow concepts:**
- **DAG:** Directed Acyclic Graph of tasks
- **Operator:** What a task does (PythonOperator, BashOperator, DatabricksOperator)
- **Sensor:** Wait for a condition (file exists, API returns 200)
- **XCom:** Pass small data between tasks
- **Connection:** Stored credentials for external systems

## Orchestration Tools Comparison

### Apache Airflow vs Lakeflow Jobs vs Prefect vs Dagster

| Feature | Airflow | Lakeflow Jobs | Prefect | Dagster |
|---------|---------|---------------|---------|---------|
| **Setup** | Self-hosted | Fully managed | Cloud/self | Cloud/self |
| **Language** | Python DAGs | YAML + UI | Python | Python |
| **Integrations** | 1000+ providers | Databricks-native | 200+ | 200+ |
| **Complexity** | Higher | Lower | Medium | Medium |
| **Flexibility** | Very high | High | Very high | Very high |
| **Asset-aware** | No | Partial | No | Yes |
| **Best for** | Multi-system | Databricks-centric | Python teams | Asset-oriented |

### When to Use Each

- **Lakeflow Jobs**: You're all-in on Databricks, want zero ops overhead
- **Airflow**: Multi-cloud, many external systems, large team already using it
- **Prefect**: Modern Python-first teams, want cloud-managed with flexibility
- **Dagster**: Asset-oriented thinking, strong data lineage requirements

### Cron Schedule Reference

```
# Cron format: minute hour day-of-month month day-of-week
# ┌───────────── minute (0-59)
# │ ┌───────────── hour (0-23)
# │ │ ┌───────────── day of month (1-31)
# │ │ │ ┌───────────── month (1-12)
# │ │ │ │ ┌───────────── day of week (0-6, Sun=0)
# │ │ │ │ │
  * * * * *

Common schedules:
  0 6 * * *     = Every day at 6 AM
  0 */4 * * *   = Every 4 hours
  0 6 * * 1-5   = Weekdays at 6 AM
  0 0 1 * *     = First day of every month
  */15 * * * *  = Every 15 minutes
```

---
# 📗 Section 7: Pipeline Metadata & Lineage

In [ ]:
import pandas as pd
from datetime import datetime
import time

class PipelineMetadataTracker:
    """Track pipeline execution metadata."""
    
    def __init__(self, pipeline_name):
        self.pipeline_name = pipeline_name
        self.run_id = f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        self.start_time = time.time()
        self.task_logs = []
    
    def log_task(self, task_name, status, rows_in=0, rows_out=0, duration_s=0):
        self.task_logs.append({
            "pipeline": self.pipeline_name,
            "run_id": self.run_id,
            "task": task_name,
            "status": status,
            "rows_in": rows_in,
            "rows_out": rows_out,
            "duration_s": round(duration_s, 2),
            "timestamp": datetime.now().isoformat()
        })
    
    def save(self, table_name="techmart_dw.pipeline_run_history"):
        """Save metadata to Delta table."""
        if self.task_logs:
            pdf = pd.DataFrame(self.task_logs)
            spark.createDataFrame(pdf).write.format("delta").mode("append").saveAsTable(table_name)
            print(f"  ✅ Saved {len(self.task_logs)} task logs to {table_name}")
    
    def summary(self):
        total_time = time.time() - self.start_time
        success = sum(1 for t in self.task_logs if t["status"] == "success")
        print(f"\n{'='*50}")
        print(f"Pipeline: {self.pipeline_name} (run: {self.run_id})")
        print(f"Tasks: {success}/{len(self.task_logs)} success")
        print(f"Duration: {total_time:.1f}s")
        print(f"{'='*50}")

# Use the tracker
tracker = PipelineMetadataTracker("daily_etl")

start = time.time()
count = spark.table("techmart_dw.orders").count()
tracker.log_task("ingest_orders", "success", rows_out=count, duration_s=time.time()-start)

start = time.time()
tracker.log_task("transform_silver", "success", rows_in=count, rows_out=count, duration_s=time.time()-start)

tracker.summary()
tracker.save()

---
# 🚀 Section 8: Mini Projects

## Project 1: Self-Healing Pipeline

In [ ]:
import time
import random

class SelfHealingPipeline:
    """Pipeline that detects failures, retries, and recovers."""
    
    def __init__(self, name, max_retries=3):
        self.name = name
        self.max_retries = max_retries
        self.tasks = []
        self.results = []
    
    def add_task(self, name, func, critical=True):
        self.tasks.append({"name": name, "func": func, "critical": critical})
    
    def run(self):
        print(f"{'='*50}")
        print(f"Self-Healing Pipeline: {self.name}")
        print(f"{'='*50}")
        
        for task in self.tasks:
            success = False
            for attempt in range(1, self.max_retries + 1):
                try:
                    result = task["func"]()
                    self.results.append({"task": task["name"], "status": "success", "attempts": attempt})
                    print(f"  ✅ {task['name']} (attempt {attempt})")
                    success = True
                    break
                except Exception as e:
                    if attempt < self.max_retries:
                        print(f"  ⚠️ {task['name']} failed (attempt {attempt}): {e}")
                        time.sleep(0.1 * attempt)
                    else:
                        self.results.append({"task": task["name"], "status": "failed", "error": str(e)})
                        print(f"  ❌ {task['name']} FAILED after {self.max_retries} attempts")
                        if task["critical"]:
                            print(f"  🛑 Critical task failed — stopping pipeline")
                            return self.results
        
        print(f"\n{'='*50}")
        success_count = sum(1 for r in self.results if r["status"] == "success")
        print(f"Complete: {success_count}/{len(self.results)} tasks succeeded")
        return self.results

# Demo
pipeline = SelfHealingPipeline("daily_orders", max_retries=3)

counter = [0]
def sometimes_fails():
    counter[0] += 1
    if counter[0] % 3 != 0:
        raise TimeoutError("Connection timeout")
    return "done"

pipeline.add_task("ingest", lambda: spark.table("techmart_dw.orders").count())
pipeline.add_task("flaky_transform", sometimes_fails, critical=True)
pipeline.add_task("write_gold", lambda: "written")
pipeline.run()

In [ ]:
# ============================================
# 🎯 YOUR TURN: Design a Pipeline Orchestration
# ============================================
# Scenario: E-commerce platform needs daily pipeline:
# Sources: orders (MySQL), inventory (PostgreSQL), events (Kafka)
#
# Requirements:
# - Orders and inventory can be ingested in parallel
# - Events must be processed after orders (for enrichment)
# - Silver layer depends on all three ingestions
# - Gold layer depends on silver
# - Quality checks run after gold
# - Report sent after quality checks pass
# - Must complete by 8 AM (started at 6 AM)
# - Retry failed tasks 3 times with 5-min delay
#
# Design:
# 1. Draw the DAG (as ASCII art)
# 2. Identify which tasks can run in parallel
# 3. What happens if quality checks fail?
# 4. How do you handle the 8 AM SLA?


In [ ]:
# ============================================
# ✅ SOLUTION
# ============================================
pipeline_design = {
    "dag_structure": """
    ingest_orders ──────────────────────────────┐
    ingest_inventory ────────────────────────────┤
    ingest_events ──── (after orders) ───────────┤
                                                  ▼
                                          transform_silver
                                                  │
                                          build_gold_tables
                                                  │
                                          quality_checks ──── FAIL → alert + stop
                                                  │
                                          send_report
    """,
    "parallel_tasks": [
        "ingest_orders and ingest_inventory run in PARALLEL (no dependency)",
        "ingest_events runs AFTER ingest_orders (needs order data for enrichment)",
    ],
    "quality_failure_handling": [
        "1. Quality checks fail → send ALERT to data team",
        "2. Mark pipeline as FAILED (do not send report with bad data)",
        "3. Page on-call engineer if failure is before 7:30 AM",
        "4. Downstream consumers see stale data (yesterday) until fixed",
        "5. Fix the issue, manually trigger pipeline, verify quality",
    ],
    "sla_management": {
        "start_time": "6:00 AM",
        "deadline": "8:00 AM",
        "monitoring": "Alert if not completed by 7:30 AM",
        "timeout": "Set 90-minute timeout on each task",
        "escalation": "Page manager if not done by 7:45 AM",
    },
}

print("PIPELINE ORCHESTRATION DESIGN")
print("=" * 60)
print("\nDAG Structure:")
print(pipeline_design["dag_structure"])
print("\nParallel Execution:")
for item in pipeline_design["parallel_tasks"]:
    print(f"  * {item}")
print("\nQuality Failure Handling:")
for step in pipeline_design["quality_failure_handling"]:
    print(f"  {step}")
print("\nSLA Management:")
for key, value in pipeline_design["sla_management"].items():
    print(f"  {key}: {value}")


---
# 🏆 Section 9: Interview Questions

## Q1: How do you handle pipeline failures in production?

1. **Retry:** Exponential backoff for transient failures (timeouts, rate limits)
2. **Alert:** Notify team via Slack/PagerDuty on persistent failures
3. **Partial recovery:** Skip failed non-critical tasks, continue with others
4. **Checkpoint:** Resume from last successful step (don't re-run everything)
5. **Dead letter:** Quarantine failed records, process rest
6. **Post-mortem:** Log full context for debugging

## Q2: Idempotent pipeline design?

- **MERGE:** Insert new, update existing — same result regardless of run count
- **Partition overwrite:** Replace entire partition — deterministic output
- **Watermark:** Only process data newer than last checkpoint
- **Test:** Run pipeline twice, compare outputs — must be identical

## Q3: Dependencies between pipelines?

1. **Shared metadata table:** Pipeline A writes "complete" flag, Pipeline B checks it
2. **Event-driven:** Pipeline A triggers Pipeline B via API/webhook
3. **Sensor pattern:** Pipeline B polls for Pipeline A's output table freshness
4. **Orchestrator:** Airflow/Workflows manages cross-pipeline dependencies

## Q4: Monitoring strategy?

- **Metrics:** Duration, record counts, error rates per task (write to Delta)
- **Alerts:** Threshold-based (duration > 2x normal, error rate > 5%)
- **Dashboard:** Pipeline health overview (last run status, freshness, trends)
- **Logs:** Structured JSON with correlation IDs for tracing

## Q5: Backfills?

1. **Parameterize:** Pipeline accepts date range as parameter
2. **Partition-level:** Reprocess specific date partitions
3. **Idempotent:** Safe to re-run any date range without duplicates
4. **Throttle:** Don't backfill all dates at once (resource contention)
5. **Validate:** Compare backfilled output to expected (regression test)

## Q6: Databricks Workflows vs Airflow?

| Feature | Databricks Workflows | Airflow |
|---|---|---|
| Setup | Zero (built-in) | Separate infrastructure |
| UI | Databricks native | Airflow web UI |
| Task types | Notebooks, SQL, DLT | Any operator |
| Multi-cloud | Databricks only | Any cloud/on-prem |
| Complexity | Simple | Complex but flexible |
| Best for | Databricks-only pipelines | Multi-system orchestration |

---
# 📗 Section 10: Advanced Orchestration Patterns

## Event-Driven Orchestration

Instead of scheduled runs, trigger pipelines when data arrives:

```python
# Lakeflow Jobs: file arrival trigger
{
    "trigger": {
        "file_arrival": {
            "url": "s3://my-bucket/raw/orders/",
            "min_time_between_triggers_seconds": 300
        }
    }
}
```

## Cross-Job Dependencies

```python
# Job B waits for Job A to complete
# Using Databricks REST API:
import requests

def wait_for_job(job_id, run_id, timeout_minutes=60):
    start = time.time()
    while time.time() - start < timeout_minutes * 60:
        response = requests.get(
            f"{DATABRICKS_HOST}/api/2.1/jobs/runs/get",
            headers={"Authorization": f"Bearer {TOKEN}"},
            params={"run_id": run_id}
        )
        state = response.json()["state"]["life_cycle_state"]
        if state == "TERMINATED":
            result = response.json()["state"]["result_state"]
            return result == "SUCCESS"
        time.sleep(30)
    return False
```

## Pipeline Monitoring with System Tables

```sql
-- Monitor job run history
SELECT
    job_id,
    run_id,
    start_time,
    end_time,
    DATEDIFF(SECOND, start_time, end_time) AS duration_seconds,
    state.result_state AS status
FROM system.lakeflow.job_run_timeline
WHERE start_time >= CURRENT_DATE() - INTERVAL 7 DAYS
ORDER BY start_time DESC;

-- Find slow jobs (taking longer than usual)
SELECT
    job_id,
    AVG(duration_seconds) AS avg_duration,
    MAX(duration_seconds) AS max_duration,
    STDDEV(duration_seconds) AS stddev_duration
FROM (
    SELECT job_id,
           DATEDIFF(SECOND, start_time, end_time) AS duration_seconds
    FROM system.lakeflow.job_run_timeline
    WHERE start_time >= CURRENT_DATE() - INTERVAL 30 DAYS
)
GROUP BY job_id
HAVING STDDEV(duration_seconds) > 300  -- High variance = unstable
ORDER BY avg_duration DESC;
```

In [ ]:
# Orchestration patterns summary
print("Orchestration Quick Reference")
print("=" * 60)

orchestration_patterns = {
    "Scheduled trigger": "0 6 * * * (cron: 6 AM daily)",
    "File arrival trigger": "When new files land in S3/ADLS/GCS",
    "Continuous trigger": "Always running (for streaming jobs)",
    "Manual trigger": "Via UI, API, or CLI",
    "Dependency trigger": "After another job completes",
}

print("\nTrigger Types:")
for trigger, description in orchestration_patterns.items():
    print(f"  {trigger}: {description}")

print("\nRetry Configuration:")
retry_config = {
    "max_retries": "3 (for transient failures)",
    "retry_delay": "5 minutes (exponential backoff)",
    "timeout": "2 hours (prevent runaway jobs)",
    "on_failure": "Email + Slack alert",
    "on_success": "Optional notification",
}
for key, value in retry_config.items():
    print(f"  {key}: {value}")

print("\nCron Schedule Examples:")
cron_examples = [
    ("0 6 * * *", "Every day at 6 AM"),
    ("0 */4 * * *", "Every 4 hours"),
    ("0 6 * * 1-5", "Weekdays at 6 AM"),
    ("0 0 1 * *", "First day of every month"),
    ("*/15 * * * *", "Every 15 minutes"),
]
for cron, description in cron_examples:
    print(f"  {cron:<20} {description}")


---
# ✅ Notebook Complete!

**What was covered:**
- Databricks Workflows (Lakeflow Jobs): tasks, dependencies, triggers
- Manual orchestration: DAGExecutor with topological sort
- Retry patterns: exponential backoff, max retries
- Idempotent design: MERGE, partition overwrite, checkpoints
- Airflow concepts: DAGs, operators, sensors, connections
- Metadata tracking: PipelineMetadataTracker writing to Delta
- Mini project: SelfHealingPipeline with retry and critical task handling
- 6 interview questions

**Next:** Notebook 26 — Data Governance & Catalog

---